In [19]:
# AutoTokenizer loads the right tokenizer for a model; AutoConfig loads the model's settings file
from transformers import AutoTokenizer, AutoConfig

In [20]:
# Only the tokenizer is downloaded here, not the 7B model weights.
# Qwen2.5-7B-Instruct is the same family we will run in 4.2.
MODEL = "Qwen/Qwen2.5-7B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)

In [21]:
# Text in, list of token IDs out. Each ID is a number in the model's vocabulary.
text = "Where is my withdrawal?"
ids = tok.encode(text)

In [22]:
# Show the text piece behind each ID. "Ġ" means a space came before the word.
pieces = tok.convert_ids_to_tokens(ids)
print(ids)
print(pieces)
print("characters:", len(text), "| tokens:", len(ids))

[9064, 374, 847, 29736, 30]
['Where', 'Ġis', 'Ġmy', 'Ġwithdrawal', '?']
characters: 23 | tokens: 5


In [23]:
# A conversation as a list of role-tagged messages, the shape the assistant service will build in 4.2
messages = [
    {"role": "system", "content": "You are ask ledger, a helpful brokerage assistant."},
    {"role": "user", "content": "Where is my withdrawal?"}
]

In [24]:
# Wrap the messages in the model's special tokens.
# tokenize=False: return readable text, not IDs.
# add_generation_prompt=True: append the marker saying the assistant's turn starts here.
formatted = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(formatted)

<|im_start|>system
You are ask ledger, a helpful brokerage assistant.<|im_end|>
<|im_start|>user
Where is my withdrawal?<|im_end|>
<|im_start|>assistant



In [25]:
# Count tokens in each part on its own, then in the full formatted prompt
system_text = messages[0]["content"]
user_text = messages[1]["content"]

system_n = len(tok.encode(system_text))
user_n = len(tok.encode(user_text))
total_n = len(tok.encode(formatted))
# What is left over is the chat template's own overhead (the special markers)
overhead_n = total_n - system_n - user_n

In [26]:
print("system:  ", system_n)
print("user:    ", user_n)
print("overhead:", overhead_n)
print("TOTAL:   ", total_n)

system:   10
user:     5
overhead: 13
TOTAL:    28


In [27]:
# The same meaning in two languages (Hindi: "Where is my withdrawal?")
english = "Where is my withdrawal?"
hindi = "मेरा निकासी कहाँ है?"

In [ ]:
# For each language, count characters and tokens, then tokens per character
for label, s in [("English", english), ("Hindi", hindi)]:
    n_tokens = len(tok.encode(s))
    n_chars = len(s)
    # Indented so it prints for both languages (it was outside the loop before)
    print(label, "| chars:", n_chars, "| tokens:", n_tokens, "| tokens/char:", round(n_tokens / n_chars, 2))
# Hindi pieces are mostly raw bytes because the vocabulary has few whole Hindi words
print(tok.convert_ids_to_tokens(tok.encode(hindi)))

In [29]:
# Read the model's real limits from its config file (no weights downloaded)
cfg = AutoConfig.from_pretrained(MODEL)
# Longest sequence the model was trained to read in one request
print("context window:", cfg.max_position_embeddings, "tokens")
# The tokenizer's own limit is a default and differs from the model's; trust the config value
print("tokenizer says:", tok.model_max_length)

context window: 32768 tokens
tokenizer says: 131072


## 4.1 findings

- "Where is my withdrawal?" = 5 tokens (23 characters)
- Chat template overhead for system + user = 13 tokens (total prompt 28)
- Same sentence in Hindi = 19 tokens vs 5 in English (about 4x)
- Qwen2.5-7B-Instruct context window = 32768 tokens
- Plan for 4.3: history budget of 6000 tokens, leaving room for the system prompt, the new message and a 512-token reply
